# 6 · The solver toolbox 🛠

Every `Inverse(...)` so far hid a whole field. This unit lays out the **standard
building blocks**: a **direct** solver, the **iterative** Krylov methods (**CG**,
**GMRes**) and the **preconditioners** that make them converge (**Jacobi**,
**block-Jacobi**, **multigrid**, **BDDC**), plus the practical levers — **threads**, a
**profile**, and **timers**. We only sketch each and point onward; the goal is to know
*what is in the box*.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from ngsolve import *
from ngsolve import solvers
import sys

if sys.platform != "emscripten":                     # JupyterLite (WebAssembly) is single-threaded
    SetNumThreads(4)                                  # use up to 4 cores (see §3)
mesh = Mesh(unit_square.GenerateMesh(maxh=0.08))
fes = H1(mesh, order=3, dirichlet=".*")
u, v = fes.TnT()
f = LinearForm(v * dx).Assemble()
print(f"test problem: Poisson, {fes.ndof} dofs")

## 1. The direct solver — and a reminder about free dofs

A **direct** solver factorises the matrix once and back-substitutes. It is robust and
exact up to round-off, and `sparsecholesky` (symmetric, positive definite) is the one
that also works in WebAssembly. Recall from unit 5 that we only invert on the **free
dofs** — the non-Dirichlet ones. Direct solvers are unbeatable until the matrix is so
large that its **factorisation** no longer fits in memory; then we go iterative.

In [ ]:
a = BilinearForm(grad(u) * grad(v) * dx).Assemble()
gfu = GridFunction(fes)
gfu.vec.data = a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky") * f.vec
print(f"direct (sparsecholesky): solution max = {max(gfu.vec):.4f}")

## 2. Iterative solvers & preconditioners

An **iterative** Krylov solver (**`CG`** for symmetric positive-definite systems,
**`GMRes`** for general ones) never forms an inverse — it only multiplies by the
matrix, and needs a cheap **preconditioner** $C\approx A^{-1}$ to converge quickly. The
better $C$ approximates $A^{-1}$, the **fewer iterations**. We compare four standard
preconditioners by the iteration count they produce on the *same* system. Several
preconditioners can be registered on one form before it is assembled.

In [ ]:
ap = BilinearForm(grad(u) * grad(v) * dx)
pre_jacobi = Preconditioner(ap, "local")                                # point Jacobi
pre_mg     = Preconditioner(ap, "multigrid", inverse="sparsecholesky")  # geometric multigrid
pre_bddc   = Preconditioner(ap, "bddc")                                 # domain decomposition
ap.Assemble()

# block-Jacobi: invert small vertex-patch blocks (all dofs of the elements at a vertex)
free = fes.FreeDofs()
blocks = []
for vtx in mesh.vertices:
    dofs = {d for el in mesh[vtx].elements for d in fes.GetDofNrs(el) if free[d]}
    if dofs:
        blocks.append(list(dofs))
pre_block = ap.mat.CreateBlockSmoother(blocks)

def cg_iterations(pre):
    n = [0]
    solvers.CG(mat=ap.mat, rhs=f.vec, sol=GridFunction(fes).vec, pre=pre, tol=1e-8,
               maxsteps=2000, printrates=False, callback=lambda *_: n.__setitem__(0, n[0] + 1))
    return n[0]

for name, pre in [("Jacobi (point)", pre_jacobi.mat), ("block-Jacobi (patches)", pre_block),
                  ("multigrid", pre_mg.mat), ("BDDC", pre_bddc.mat)]:
    print(f"  CG + {name:24s}: {cg_iterations(pre):4d} iterations")

The pattern is universal: **point-Jacobi** is cheap but weak (many iterations);
**block-Jacobi** groups locally-coupled dofs and does better; **multigrid** and
**BDDC** are *scalable* — their iteration count barely grows as the mesh is refined,
which is exactly what large problems demand. For a **non-symmetric** system (e.g. added
convection) `CG` is invalid; **`GMRes`** takes over with the same preconditioner idea.

In [ ]:
wind = CF((20, 0))                                   # a convection term breaks symmetry
anonsym = BilinearForm(grad(u) * grad(v) * dx + (wind * grad(u)) * v * dx)
pre_ns = Preconditioner(anonsym, "local")
anonsym.Assemble()
gns = GridFunction(fes)
solvers.GMRes(A=anonsym.mat, b=f.vec, x=gns.vec, pre=pre_ns.mat, tol=1e-8,
              maxsteps=1000, printrates=False)
print(f"GMRes (non-symmetric convection-diffusion): max = {max(gns.vec):.4f}")

## 3. Practical levers — threads, a profile, and timers

Two knobs decide *how fast* a solver runs. **`SetNumThreads(n)`** sets the task
parallelism and a **`TaskManager`** block farms the work across those threads. Passing
**`pajetrace=…`** records a **timeline** of *exactly that block* — NGSolve writes a small,
self-contained HTML with an interactive **sunburst**, which we embed right below. The
**`Timers()`** list underneath describes the **same** block: since `Timers()` is
*cumulative* over the whole session (meshing, all the solves above, …), we snapshot it
**before and after** and report only the **difference** — otherwise the two would not
agree. *(Threads and tracing need a real OS — in single-threaded JupyterLite this is skipped.)*

In [ ]:
import glob, os, html, pathlib
from IPython.display import display, HTML

if sys.platform != "emscripten":                     # threads/tracing unavailable in JupyterLite
    SetNumThreads(4)
    before = {t["name"]: t["time"] for t in Timers()}    # snapshot to scope the timers
    with TaskManager(pajetrace=10**8):                   # the sunburst covers exactly this block
        for _ in range(5):
            BilinearForm(grad(u) * grad(v) * dx).Assemble()
    traces = sorted(glob.glob("ng*.html"), key=os.path.getmtime)   # the viewer NGSolve just wrote
    if traces:
        doc = html.escape(pathlib.Path(traces[-1]).read_text(), quote=True)
        display(HTML(f'<iframe srcdoc="{doc}" sandbox="allow-scripts" width="100%" height="460" '
                     f'style="border:1px solid #ddd;border-radius:8px" '
                     f'title="pajetrace sunburst"></iframe>'))
    # the same block through NGSolve's per-routine timers (difference, since Timers() is cumulative):
    delta = sorted(((t["time"] - before.get(t["name"], 0.0), t["name"]) for t in Timers()), reverse=True)
    print("hottest routines in the profiled block (same scope as the sunburst):")
    for dt, name in delta[:5]:
        if dt > 0:
            print(f"  {dt * 1e3:8.2f} ms  {name[:46]}")
else:
    print("(threaded profiling / pajetrace needs a real OS — run locally or on Colab)")

That closes **Part I**: you can give the Beast geometry, functions, spaces, weak forms
and now the solvers to crack them at scale. Time to climb on — **into the saddle**, and
the features that real applications need.

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("05-solving", "5 · Solving — Dirichlet dofs & static condensation")
    _next = ("07-saddle-point", "7 · Mixed problems — the saddle point 🐎")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))